# Stochastic fm loss

test `nami.stochastic_fm_loss` with the class-based gamma schedules and verify that `ZeroGamma()` recovers the deterministic `nami.fm_loss`.

In [1]:
import torch
from torch import nn
import nami

In [3]:
batch, dim = 8, 4
x_source = torch.randn(batch, dim)
x_target = torch.randn(batch, dim)
t = torch.rand(batch)
z = torch.randn_like(x_target)

In [4]:
vfield = nami.VelocityField(dim=4)

In [5]:
loss_det = nami.regression_loss(
    vfield,
    x_noise=x_source,
    x_data=x_target,
    interpolant=nami.LinearInterpolant(),
    parameterization=nami.velocity_prediction(),
)

loss_stoch = nami.losses.stochastic_fm.stochastic_fm_loss(
    vfield,
    x_noise=x_source,
    x_data=x_target,
    gamma=nami.BrownianGamma(),
    z=z,
    reduction="mean",
)

print(f"deterministic fm_loss: {loss_det.item():.6f}")
print(f"stochastic_fm_loss (Brownian): {loss_stoch.item():.6f}")

deterministic fm_loss: 2.610054
stochastic_fm_loss (Brownian): 3.514440


In [ ]:
loss_det_none = nami.regression_loss(
    vfield,
    x_noise=x_source,
    x_data=x_target,
    interpolant=nami.LinearInterpolant(),
    parameterization=nami.velocity_prediction(),
    reduction="none",
)

loss_zero_none = nami.stochastic_fm_loss(
    vfield,
    x_noise=x_source,
    x_data=x_target,
    gamma=nami.ZeroGamma(),
    z=z,
    reduction="none",
)

print("ZeroGamma matches deterministic fm_loss?", torch.allclose(loss_det_none, loss_zero_none, atol=1e-6))

ZeroGamma matches deterministic fm_loss: False


Note that in both cases we use different t distributions, `regression_loss` defaults to eps_t=1e-3 (samples t ~ U[1e-3, 1-1e-3]). `stochastic_fm_loss` forwards eps_t=0.0 (samples t ~ U[0, 1]).

In [9]:
torch.manual_seed(0)
t = torch.rand(x_source.shape[0])

loss_det_none = nami.regression_loss(
    vfield,
    x_noise=x_source,
    x_data=x_target,
    t=t,
    interpolant=nami.LinearInterpolant(),
    parameterization=nami.velocity_prediction(),
    eps_t=0.0,
    reduction="none",
)

loss_zero_none = nami.stochastic_fm_loss(
    vfield,
    x_noise=x_source,
    x_data=x_target,
    t=t,
    gamma=nami.ZeroGamma(),
    z=z,
    reduction="none",
)

print("ZeroGamma matches deterministic fm_loss:",
    torch.allclose(loss_det_none, loss_zero_none, atol=1e-6))

ZeroGamma matches deterministic fm_loss: True


In [ ]:
optimizer = torch.optim.Adam(vfield.parameters(), lr=1e-3)

for step in range(10):
    loss = nami.stochastic_fm_loss(
        vfield,
        x_noise=x_source,
        x_data=x_target,
        gamma=nami.ScaledBrownianGamma(scale=1.1),
        reduction="mean",
    )
    print(f"step={step} loss={loss.item():.6f}")
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"step={step} loss={loss.item():.6f}")

step=0 loss=14.522295
step=0 loss=14.522295
step=1 loss=54.836300
step=1 loss=54.836300
step=2 loss=2.068749
step=2 loss=2.068749
step=3 loss=8.345240
step=3 loss=8.345240
step=4 loss=2.306889
step=4 loss=2.306889
step=5 loss=3.899499
step=5 loss=3.899499
step=6 loss=2.809108
step=6 loss=2.809108
step=7 loss=7.826230
step=7 loss=7.826230
step=8 loss=3.007683
step=8 loss=3.007683
step=9 loss=2.517387
step=9 loss=2.517387
